In [1]:
import mlflow
from mlflow.tracking import MlflowClient


In [2]:
mlflow.set_tracking_uri("http://localhost:5001")

In [3]:
mlclient = MlflowClient()

In [4]:
run_id = "5c67fb931b5148c1aa0b1bcb15a4a328"

In [7]:
run = mlclient.get_run(run_id)
last_checkpoint_step = run.data.tags.get("last_checkpoint_step")
last_checkpoint_step

'5500'

In [8]:
latest_checkpoint_folder = f'checkpoint_{last_checkpoint_step}'
latest_checkpoint_folder

'checkpoint_5500'

In [9]:
model_path = mlclient.download_artifacts(run_id, f"{latest_checkpoint_folder}/artifacts/model.zip")

In [10]:
model_path

'C:\\Users\\coool\\AppData\\Local\\Temp\\tmpkcw4pft9\\checkpoint_5500\\artifacts/model.zip'

In [5]:
model_artifacts = mlclient.list_artifacts(run_id, path='model')   

In [6]:
model_artifacts

[]

In [6]:
from trading_functions.db.session import SessionLocal
from sqlalchemy.orm import Session

From inside trading_functions, user :  cooolrahul_postgres
Using Postgres DB: INF_DB at localhost:5432 with user cooolrahul_postgres


In [7]:
from rl_functions.utils import get_closest_trading_date

In [8]:
import datetime

In [9]:
start_date = datetime.datetime.strptime('2023-01-01', "%Y-%m-%d").date()
start_date

datetime.date(2023, 1, 1)

In [10]:
from dotenv import load_dotenv

In [11]:
load_dotenv("../.env_local", override=True)

True

In [12]:
import os
os.getenv("POSTGRES_HOST")

'localhost'

In [13]:
db: Session = SessionLocal()

In [14]:
first_date = get_closest_trading_date(db=db, 
                         symbol='SPY',
                         target_date=start_date
                         )

In [19]:
first_date

(datetime.datetime(2023, 1, 3, 9, 30),)

In [22]:
second_date = get_closest_trading_date(db=db, 
                         symbol='SPY',
                         target_date=first_date.time.date() + datetime.timedelta(days=1),
                         only_next=True
                         )

In [23]:
second_date

(datetime.datetime(2023, 1, 4, 9, 30),)

### Testing the issue with get data starting from 11 AM

In [ ]:
def get_env(config: dict, db: Session, eval_mode: bool=False, dagster_logger=None):
    """
    Creates and returns a DummyVecEnv wrapped TradingEnv instance based on the provided configuration and database session.
    If eval_mode is True, it sets up the environment for evaluation using test data; otherwise, it sets up for training using training data.
    """
    from rl_functions.trading_env import TradingEnv

    obs_features = get_observation_features(config)
    start_date_config_key = 'train_start_date' if not eval_mode else 'test_start_date'
    end_date_config_key = 'train_end_date' if not eval_mode else 'test_end_date' 
    start_date = datetime.datetime.strptime(config['rl'][start_date_config_key], "%Y-%m-%d").date()
    if not eval_mode:
        start_date_first = get_closest_trading_date(db=db, symbol='SPY', target_date=start_date, only_next=True).time.date()
        start_date = get_closest_trading_date(db=db, symbol='SPY', target_date=start_date_first + datetime.timedelta(days=1), only_next=True).time.date()
    end_date = datetime.datetime.strptime(config['rl'][end_date_config_key], "%Y-%m-%d").date()
    initial_balance = float(config['rl']['initial_balance'])
    trade_fee = float(config['rl']['trade_fee'])
    max_trade_loss_percent = float(config['rl']['max_trade_loss_percent'])
    price_multiplier = float(config['rl']['price_multiplier'])


    def make_env():
        return TradingEnv(db=db,
                          symbol='SPY',
                          start_date=start_date,
                          end_date=end_date,
                          initial_balance=initial_balance,
                          trade_fee=trade_fee,
                          max_trade_loss_percent=max_trade_loss_percent,
                          obs_features=obs_features,
                          price_multiplier=price_multiplier,
                            evaluation=eval_mode,
                            logger=dagster_logger
                          )
    
    return make_env

In [32]:
from rl_functions.utils import get_env
import yaml
import logging
import mlflow
from mlflow.tracking import MlflowClient

In [33]:
mlflow.set_tracking_uri("http://localhost:5001")

In [42]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger(__name__)

In [43]:
logger.info("Starting test_mlflow_download_model")

[2026-02-20 20:57:51,588] INFO __main__: Starting test_mlflow_download_model


In [38]:
db: Session = SessionLocal()

In [39]:
config_path = "../Config/config_dev.yaml"

In [40]:
with open(config_path, 'r') as file:
    conf = yaml.safe_load(file)
            

In [44]:

eval_env = get_env(
            config=conf,
            db=db,
            eval_mode=True,
            dagster_logger=logger)

[2026-02-20 20:58:04,948] INFO __main__: Initializing TradingEnv for symbol: SPY, start_date: 2023-04-03, end_date: 2023-04-09, evaluation: True
[2026-02-20 20:58:04,953] INFO rl_functions.utils: Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev.yaml
[2026-02-20 20:58:04,970] INFO rl_functions.utils: Loading inference configuration from C:/Projects/Trading/Uns_SPY_Trading/Config/config_dev.yaml
[2026-02-20 20:58:04,985] INFO root: Using default MLflow tracking URI from config : http://localhost:5001
[2026-02-20 20:58:04,988] INFO root: Using model alias: rl for model: xgboost_rob_2023_jan_high
[2026-02-20 20:58:05,045] INFO root: scaler path: run_model/scalers.pkl
[2026-02-20 20:58:05,046] INFO root: config artifact path: run_model/training_config.yaml
[2026-02-20 20:58:05,048] INFO root: Run ID: 3c97bd2da876481d978de23997d0c42e
[2026-02-20 20:58:05,049] INFO root: Deleting existing scaler directory: /model_local_artifacts


Evaluation mode: ON


[2026-02-20 20:58:05,233] INFO root: Downloading artifacts to /model_local_artifacts


run_model/scalers.pkl  (dir: False)
run_model/training_config.yaml  (dir: False)


[2026-02-20 20:58:05,315] INFO root: Scalers downloaded to c:\model_local_artifacts\scalers.pkl
[2026-02-20 20:58:05,316] INFO root: Scalers loaded: dict_keys(['minmax', 'standard', 'robust'])
[2026-02-20 20:58:05,414] INFO root: Config downloaded to c:\model_local_artifacts\training_config.yaml
2026/02/20 20:58:05 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pandas (current: 2.3.3, required: pandas==2.3.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/02/20 20:58:05 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pandas (current: 2.3.3, required: pandas==2.3.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's en

start_idx: 100


[2026-02-20 20:58:08,490] INFO root: Data shape after adding indicators: (391, 74)
[2026-02-20 20:58:08,498] INFO root: Data shape before scaling: (391, 74)
[2026-02-20 20:58:09,199] INFO __main__: Data shape : (291, 94)
[2026-02-20 20:58:09,213] INFO __main__: Data time range : 2023-04-04 11:09:00 to 2023-04-04 15:59:00


In [45]:
eval_env.data.head()

AttributeError: 'DummyVecEnv' object has no attribute 'data'